# Enhanced Mental Health Data Preprocessing Pipeline
Handles different dataset structures with robust file detection and processing



In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import pickle
import h5py
import librosa
from sklearn.preprocessing import StandardScaler
from transformers import BertTokenizer, BertModel
import opensmile
import glob
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
class FixedFeatureExtractor:
  """
    Extracts features for all 4 fixed modalities:
    - Text (T): BERT features (768D)
    - Audio (A): eGeMAPSv02 features (88D)
    - Video (V): ResNet50 features (2048D)
    - Physiological (P): ECG features (64D)
    """

    def __init__(self):
        self.setup_models()

    def setup_models(self):
        """Initialize models with better error handling"""
        print("Loading feature extraction models...")

        # BERT for text
        try:
            self.bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
            self.bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)
            self.bert_model.eval()
            print("✓ BERT loaded successfully")
        except Exception as e:
            print(f"❌ BERT failed: {e}")
            self.bert_tokenizer = None
            self.bert_model = None

        # OpenSMILE for audio
        try:
            self.smile = opensmile.Smile(
                feature_set=opensmile.FeatureSet.eGeMAPSv02,
                feature_level=opensmile.FeatureLevel.Functionals,
            )
            print("✓ OpenSMILE loaded successfully")
        except Exception as e:
            print(f"❌ OpenSMILE failed: {e}")
            self.smile = None

    def extract_text_features(self, text):
        """Extract BERT features with validation"""
        if not self.bert_model or not isinstance(text, str) or len(text.strip()) < 3:
            return torch.zeros(768)

        try:
            # Clean text
            text = str(text).strip()[:512]  # Limit length

            inputs = self.bert_tokenizer(
                text, return_tensors='pt', max_length=512,
                truncation=True, padding=True
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = self.bert_model(**inputs)
                features = outputs.last_hidden_state[:, 0, :].cpu()

            return features.squeeze(0)
        except Exception as e:
            print(f"Text extraction error: {e}")
            return torch.zeros(768)

    def extract_audio_features(self, audio_data, sample_rate=16000):
        """Extract audio features with fallback"""
        try:
            if self.smile and audio_data is not None and len(audio_data) > 0:
                if isinstance(audio_data, torch.Tensor):
                    audio_data = audio_data.numpy()

                # Ensure proper format
                if len(audio_data.shape) > 1:
                    audio_data = audio_data.mean(axis=0)

                # Resample if needed
                if sample_rate != 16000 and len(audio_data) > 0:
                    audio_data = librosa.resample(audio_data, orig_sr=sample_rate, target_sr=16000)

                features = self.smile.process_signal(audio_data, 16000)
                return torch.tensor(features.values[0], dtype=torch.float32)
        except Exception as e:
            print(f"Audio extraction error: {e}")

        return torch.zeros(88)

In [ ]:
try:
    from mmsdk import mmdatasdk
    print("CMU-MultimodalSDK imported successfully at top level.")
except ImportError:
    print("CMU-MultimodalSDK not found at top level. Ensure it's installed and path is set.")
    print("Processing of CSD files will fail unless installed correctly.")


class FixedDatasetLoader:
    def __init__(self, feature_extractor):
        self.feature_extractor = feature_extractor

    def load_depression_dataset(self, data_path):
        """Fixed depression dataset loader"""
        print(f"Loading depression dataset from: {data_path}")
        samples = []

        # Find CSV files
        csv_patterns = [
            "dataset.csv", "depression.csv", "*.csv",
            "**/dataset.csv", "**/*.csv"
        ]

        csv_files = []
        for pattern in csv_patterns:
            found_files = glob.glob(os.path.join(data_path, pattern), recursive=True)
            csv_files.extend(found_files)

        if not csv_files:
            print(f"❌ No CSV files found in {data_path}")
            return samples

        for csv_file in csv_files:
            print(f"Processing: {csv_file}")
            try:
                # Try different encodings
                df = None
                for encoding in ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']:
                    try:
                        df = pd.read_csv(csv_file, encoding=encoding)
                        print(f"✓ Loaded with {encoding} encoding")
                        break
                    except:
                        continue

                if df is None:
                    print(f"❌ Could not load {csv_file}")
                    continue

                print(f"Shape: {df.shape}, Columns: {list(df.columns)}")

                # Remove completely empty rows
                df = df.dropna(how='all')

                # Find text and label columns
                text_col = self._find_column(df.columns, ['Tweets', 'text', 'content', 'post', 'message'])
                label_col = self._find_column(df.columns, ['Labels', 'label', 'class', 'sentiment', 'target'])

                if not text_col or not label_col:
                    print(f"❌ Required columns not found. Text: {text_col}, Label: {label_col}")
                    continue

                # Process valid rows
                valid_count = 0
                for idx, row in df.iterrows():
                    try:
                        text = str(row[text_col]).strip()
                        label_val = row[label_col]

                        # Skip if text is too short or label is invalid
                        if len(text) < 5 or pd.isna(label_val):
                            continue

                        # Map labels (0: not depressed, 1: mildly depressed, 2: severely depressed)
                        if isinstance(label_val, str):
                            label_map = {
                                '0': 0, 'no': 0, 'none': 0, 'normal': 0,
                                '1': 1, 'mild': 1, 'moderate': 1,
                                '2': 2, 'severe': 2, 'yes': 2
                            }
                            mapped_label = label_map.get(str(label_val).lower(), 1)
                        else:
                            mapped_label = max(0, min(2, int(float(label_val))))

                        # Extract features
                        text_feat = self.feature_extractor.extract_text_features(text)

                        samples.append({
                            'user_id': f"dep_user_{idx}",
                            'text': text_feat,
                            'audio': torch.zeros(88), # Placeholder for audio features
                            'video': torch.zeros(2048), # Placeholder for video features
                            'label': mapped_label,
                            'source': 'depression'
                        })
                        valid_count += 1

                    except Exception as e:
                        print(f"Error processing row {idx}: {e}")
                        continue

                print(f"✓ Processed {valid_count} valid samples from {csv_file}")

            except Exception as e:
                print(f"❌ Error with {csv_file}: {e}")
                continue

        print(f"✓ Total depression samples: {len(samples)}")
        return samples

    def load_bipolar_dataset(self, data_path):
        """Fixed bipolar dataset loader"""
        print(f"Loading bipolar dataset from: {data_path}")
        samples = []

        # Find all possible files
        file_patterns = [
            "*.csv", "*.txt", "*.json", "**/*.csv", "**/*.txt"
        ]

        all_files = []
        for pattern in file_patterns:
            found_files = glob.glob(os.path.join(data_path, pattern), recursive=True)
            all_files.extend(found_files)

        if not all_files:
            print(f"❌ No files found in {data_path}")
            return samples

        print(f"Found {len(all_files)} potential files")

        for file_path in all_files:
            print(f"Trying: {file_path}")

            if file_path.endswith('.csv'):
                try:
                    # Try multiple separators and encodings
                    df = None
                    for sep in [',', '\t', ';', '|']:
                        for encoding in ['utf-8', 'latin-1', 'cp1252']:
                            try:
                                df = pd.read_csv(file_path, sep=sep, encoding=encoding)
                                if df.shape[1] > 1:  # Valid if more than 1 column
                                    print(f"✓ Loaded with sep='{sep}', encoding='{encoding}'")
                                    break
                            except:
                                continue
                        if df is not None and df.shape[1] > 1:
                            break

                    if df is None or df.shape[1] <= 1:
                        print(f"❌ Could not parse {file_path}")
                        continue

                    print(f"Shape: {df.shape}, Columns: {list(df.columns)}")

                    # Find relevant columns
                    text_col = self._find_column(df.columns,
                        ['text', 'post', 'tweet', 'content', 'message', 'statement'])
                    label_col = self._find_column(df.columns,
                        ['label', 'mood', 'state', 'class', 'sentiment', 'emotion'])

                    if not text_col:
                        print(f"❌ No text column found")
                        continue

                    # Process data
                    valid_count = 0
                    for idx, row in df.iterrows():
                        try:
                            text = str(row[text_col]).strip()

                            if len(text) < 5:
                                continue

                            # Handle labels
                            if label_col:
                                label_val = row[label_col]
                                if isinstance(label_val, str):
                                    mood_map = {
                                        'depressed': 0, 'depression': 0, 'low': 0, 'sad': 0,
                                        'normal': 1, 'neutral': 1, 'stable': 1, 'ok': 1,
                                        'manic': 2, 'mania': 2, 'high': 2, 'elevated': 2, 'happy': 2
                                    }
                                    mapped_label = mood_map.get(str(label_val).lower(), 1)
                                else:
                                    mapped_label = max(0, min(2, int(float(label_val))))
                            else:
                                mapped_label = 1  # Default to neutral

                            # Extract features
                            text_feat = self.feature_extractor.extract_text_features(text)

                            samples.append({
                                'user_id': f"bp_user_{idx}",
                                'text': text_feat,
                                'audio': torch.zeros(88), # Placeholder for audio features
                                'video': torch.zeros(2048), # Placeholder for video features
                                'label': mapped_label,
                                'source': 'bipolar'
                            })
                            valid_count += 1

                        except Exception as e:
                            print(f"Error processing row {idx}: {e}")
                            continue

                    print(f"✓ Processed {valid_count} samples from {file_path}")

                except Exception as e:
                    print(f"❌ Error processing {file_path}: {e}")
                    continue

        print(f"✓ Total bipolar samples: {len(samples)}")
        return samples

    def load_wesad_dataset(self, data_path):
        """Enhanced WESAD dataset loader"""
        print(f"Loading WESAD dataset from: {data_path}")
        samples = []

        # Find subject files
        subject_patterns = ["S*.pkl", "**/S*.pkl", "*.pkl", "**/*.pkl"]
        subject_files = []

        for pattern in subject_patterns:
            found_files = glob.glob(os.path.join(data_path, pattern), recursive=True)
            subject_files.extend(found_files)

        if not subject_files:
            print(f"❌ No subject files found in {data_path}")
            return samples

        print(f"Found {len(subject_files)} potential WESAD files")

        for subject_file in subject_files:
            try:
                print(f"Processing: {subject_file}")

                with open(subject_file, 'rb') as f:
                    try:
                        subject_data = pickle.load(f, encoding='latin1')
                    except:
                        try:
                            subject_data = pickle.load(f)
                        except:
                            print(f"❌ Could not load {subject_file}")
                            continue

                if not isinstance(subject_data, dict):
                    print(f"❌ Invalid data format in {subject_file}")
                    continue

                print(f"Data keys: {list(subject_data.keys())}")

                # Extract signals and labels
                signal_data = subject_data.get('signal', {})
                labels = subject_data.get('label', [])

                if len(labels) == 0:
                    print(f"❌ No labels found in {subject_file}")
                    continue

                # Process chest signals if available
                chest_data = signal_data.get('chest', {})

                # Create segments
                segment_length = 1000  # Adjust based on data
                step_size = 500

                valid_count = 0
                for i in range(0, len(labels) - segment_length, step_size):
                    try:
                        segment_labels = labels[i:i+segment_length]
                        dominant_label = max(set(segment_labels), key=list(segment_labels).count)

                        # WESAD labels: 0=not defined, 1=baseline, 2=stress, 3=amusement, 4=meditation
                        if dominant_label not in [1, 2, 3, 4]:
                            continue

                        # Map to 3-class: 1->0 (calm), 2->1 (stress), 3,4->2 (other)
                        label_map = {1: 0, 2: 1, 3: 2, 4: 2}
                        mapped_label = label_map[dominant_label]

                        # Create pseudo-audio from physiological signals
                        audio_feat = torch.zeros(88)
                        if 'ECG' in chest_data and len(chest_data['ECG']) > i+segment_length:
                            ecg_segment = chest_data['ECG'][i:i+segment_length]
                            if len(ecg_segment) > 0:
                                try:
                                    from scipy import signal
                                    ecg_resampled = signal.resample(ecg_segment, 16000)
                                    audio_feat = self.feature_extractor.extract_audio_features(ecg_resampled)
                                except:
                                    audio_feat = torch.zeros(88)

                        samples.append({
                            'user_id': os.path.basename(subject_file).replace('.pkl', f'_seg_{i}'),
                            'text': torch.zeros(768), # Placeholder as WESAD is not text-based
                            'audio': audio_feat,
                            'video': torch.zeros(2048), # Placeholder as WESAD is not video-based
                            'label': mapped_label,
                            'source': 'wesad'
                        })
                        valid_count += 1

                    except Exception as e:
                        print(f"Error processing segment {i}: {e}")
                        continue

                print(f"✓ Processed {valid_count} segments from {subject_file}")

            except Exception as e:
                print(f"❌ Error with {subject_file}: {e}")
                continue

        print(f"✓ Total WESAD samples: {len(samples)}")
        return samples

    def load_cmu_mosei_dataset(self, data_path):
        """Loads and processes CMU-MOSEI dataset using CMU-MultimodalSDK."""
        print(f"Loading CMU-MOSEI dataset from: {data_path}")
        samples = []

        try:
            # Ensure mmsdk is imported. If not available at top, try importing here again.
            # This is a fallback if the top-level import fails, but best to fix top-level.
            try:
                from mmsdk import mmdatasdk
            except ImportError:
                print("❌ CMU-MultimodalSDK not found. Please ensure it is installed correctly to process CSD files.")
                print("You might need to run the following commands in a separate cell at the beginning of your notebook:")
                print("!git clone https://github.com/CMU-MultiComp-Lab/CMU-MultimodalSDK.git")
                print("%cd CMU-MultimodalSDK")
                print("!pip install -r requirements.txt")
                print("!pip install .")
                print("import sys")
                print("sys.path.append(\"/content/CMU-MultimodalSDK\")")
                return samples # Exit if SDK is not found

            print("Attempting to load CMU-MOSEI dataset with CMU-MultimodalSDK...")

            # Load the entire dataset
            # This expects the 'data_path' to be the root directory containing CSD files
            cmu_mosei = mmdatasdk.mmdataset(data_path)
            print("✓ CMU-MOSEI dataset initialized successfully using mmdatasdk.")

            # Correctly reference the label computational sequence name
            if 'All Labels' not in cmu_mosei.computational_sequences:
                print("❌ 'All Labels' not found in MOSEI dataset. Cannot process.")
                return samples

            video_ids = list(cmu_mosei.computational_sequences['All Labels'].keys())
            print(f"Found {len(video_ids)} video IDs in MOSEI.")

            valid_count = 0
            for vid_id in video_ids:
                try:
                    # Get labels from 'All Labels' computational sequence
                    label_seq = cmu_mosei.computational_sequences['All Labels'][vid_id]['features']
                    if len(label_seq) == 0:
                        continue

                    # CMU-MOSEI labels are typically in the range [-3, 3] for sentiment
                    # We need to map them to 0 (negative), 1 (neutral), 2 (positive)
                    sentiment_score = float(label_seq[0][0]) # Assuming first score in the sequence

                    if sentiment_score < -0.5: # Adjust thresholds as needed
                        mapped_label = 0  # Negative
                    elif sentiment_score > 0.5:
                        mapped_label = 2  # Positive
                    else:
                        mapped_label = 1  # Neutral

                    # --- IMPORTANT: Feature Extraction from MOSEI CSDs ---
                    # This is where you'd extract the actual features using your feature_extractor
                    # Here's a conceptual outline for how you'd get the actual features:

                    text_feat = torch.zeros(768) # Default to zeros
                    if 'CMU_MOSEI_TimestampedWordVectors' in cmu_mosei.computational_sequences and \
                       vid_id in cmu_mosei.computational_sequences['CMU_MOSEI_TimestampedWordVectors']:
                        # The SDK provides word embeddings. To get 'text' for your BERT extractor,
                        # you would need to retrieve the original words from the dataset
                        # (which is a separate download, usually).
                        # For now, we'll keep it as a placeholder. If you have the raw text,
                        # you would feed it to self.feature_extractor.extract_text_features(raw_text).
                        # Example: word_vectors = cmu_mosei.computational_sequences['CMU_MOSEI_TimestampedWordVectors'][vid_id]['features']
                        # You might need to re-construct text from these if raw text isn't available.
                        pass

                    audio_feat = torch.zeros(88) # Default to zeros
                    if 'CMU_MOSEI_COVAREP' in cmu_mosei.computational_sequences and \
                       vid_id in cmu_mosei.computational_sequences['CMU_MOSEI_COVAREP']:
                        covarep_features = cmu_mosei.computational_sequences['CMU_MOSEI_COVAREP'][vid_id]['features']
                        # COVAREP features are already extracted audio features.
                        # You might average them or use a more sophisticated pooling.
                        if len(covarep_features) > 0:
                            # Assuming covarep_features is a 2D array (time, feature_dim)
                            audio_feat = torch.tensor(np.mean(covarep_features, axis=0), dtype=torch.float32)
                            # Resize if feature_extractor expects a different size
                            if audio_feat.shape[0] != 88:
                                # Simple truncation/padding for demonstration
                                if audio_feat.shape[0] > 88:
                                    audio_feat = audio_feat[:88]
                                else:
                                    audio_feat = torch.cat([audio_feat, torch.zeros(88 - audio_feat.shape[0])])


                    video_feat = torch.zeros(2048) # Default to zeros
                    if 'CMU_MOSEI_VisualOpenFace2' in cmu_mosei.computational_sequences and \
                       vid_id in cmu_mosei.computational_sequences['CMU_MOSEI_VisualOpenFace2']:
                        openface_features = cmu_mosei.computational_sequences['CMU_MOSEI_VisualOpenFace2'][vid_id]['features']
                        if len(openface_features) > 0:
                            video_feat = torch.tensor(np.mean(openface_features, axis=0), dtype=torch.float32)
                            # Resize if needed
                            if video_feat.shape[0] != 2048:
                                if video_feat.shape[0] > 2048:
                                    video_feat = video_feat[:2048]
                                else:
                                    video_feat = torch.cat([video_feat, torch.zeros(2048 - video_feat.shape[0])])
                    elif 'CMU_MOSEI_VisualFacet42' in cmu_mosei.computational_sequences and \
                         vid_id in cmu_mosei.computational_sequences['CMU_MOSEI_VisualFacet42']:
                        facet_features = cmu_mosei.computational_sequences['CMU_MOSEI_VisualFacet42'][vid_id]['features']
                        if len(facet_features) > 0:
                            video_feat = torch.tensor(np.mean(facet_features, axis=0), dtype=torch.float32)
                            # Resize if needed
                            if video_feat.shape[0] != 2048:
                                if video_feat.shape[0] > 2048:
                                    video_feat = video_feat[:2048]
                                else:
                                    video_feat = torch.cat([video_feat, torch.zeros(2048 - video_feat.shape[0])])
                    # --- End Feature Extraction ---

                    samples.append({
                        'user_id': str(vid_id),
                        'text': text_feat,
                        'audio': audio_feat,
                        'video': video_feat,
                        'label': mapped_label,
                        'source': 'mosei'
                    })
                    valid_count += 1

                except Exception as e:
                    print(f"Error processing video ID {vid_id}: {e}")
                    continue

            print(f"✓ Processed {valid_count} valid samples from CMU-MOSEI.")

        except Exception as e:
            print(f"❌ Error loading CMU-MOSEI dataset with mmdatasdk: {e}")

        print(f"✓ Total CMU-MOSEI samples: {len(samples)}")
        return samples

    def _process_mosei_pickle(self, data, file_path):
        """Process MOSEI pickle data - kept for completeness but CSD is primary for MOSEI"""
        samples = []

        try:
            if isinstance(data, dict):
                for video_id, video_data in data.items():
                    if isinstance(video_data, dict):
                        text = video_data.get('text', video_data.get('words', ''))
                        audio = video_data.get('audio', video_data.get('covarep', None))
                        video = video_data.get('vision', video_data.get('facet', None))
                        labels = video_data.get('label', video_data.get('labels', None))

                        if labels is None:
                            continue

                        if isinstance(text, list):
                            text_str = ' '.join([str(t) for t in text if t])
                        else:
                            text_str = str(text)

                        text_feat = self.feature_extractor.extract_text_features(text_str)

                        audio_feat = torch.zeros(88)
                        if audio is not None:
                            try:
                                if isinstance(audio, (list, np.ndarray, torch.Tensor)):
                                    audio_array = np.array(audio)
                                    if len(audio_array.shape) > 1:
                                        audio_array = audio_array.mean(axis=0)
                                    audio_feat = self.feature_extractor.extract_audio_features(audio_array)
                            except:
                                audio_feat = torch.zeros(88)

                        video_feat = torch.zeros(2048)

                        if isinstance(labels, (list, np.ndarray)):
                            label_val = float(labels[0]) if len(labels) > 0 else 0.0
                        else:
                            label_val = float(labels)

                        if label_val < -1.0:
                            mapped_label = 0
                        elif label_val > 1.0:
                            mapped_label = 2
                        else:
                            mapped_label = 1

                        samples.append({
                            'user_id': str(video_id),
                            'text': text_feat,
                            'audio': audio_feat,
                            'video': video_feat,
                            'label': mapped_label,
                            'source': 'mosei'
                        })

        except Exception as e:
            print(f"Error processing MOSEI pickle data: {e}")

        return samples

    def _read_h5_recursively(self, h5_file):
        """Recursively read HDF5 file"""
        data = {}

        def visit_func(name, obj):
            if isinstance(obj, h5py.Dataset):
                try:
                    data[name] = obj[()]
                except:
                    data[name] = None

        h5_file.visititems(visit_func)
        return data

    def _process_mosei_h5(self, data, file_path):
        """Process MOSEI HDF5 data - placeholder"""
        samples = []
        print(f"Note: HDF5 processing for MOSEI is not fully implemented and uses placeholder logic.")
        # Example: If your HDF5 has 'data' and 'labels' keys, you'd process them here
        if 'data' in data and 'labels' in data:
            for i in range(len(data['labels'])):
                # Assuming data['data'][i] is text, and data['labels'][i] is a single label
                text_feat = self.feature_extractor.extract_text_features(str(data['data'][i]))
                mapped_label = max(0, min(2, int(float(data['labels'][i]))))
                samples.append({
                    'user_id': f"mosei_h5_{i}",
                    'text': text_feat,
                    'audio': torch.zeros(88),
                    'video': torch.zeros(2048),
                    'label': mapped_label,
                    'source': 'mosei'
                })
        return samples

    def _find_column(self, columns, keywords):
        """Find column matching keywords"""
        columns_lower = [col.lower() for col in columns]
        for keyword in keywords:
            for col, col_lower in zip(columns, columns_lower):
                if keyword.lower() in col_lower:
                    return col
        return None

CMU-MultimodalSDK imported successfully at top level.


In [ ]:
def run_fixed_preprocessing_pipeline():
    """Main preprocessing function"""

    # Dataset paths - UPDATE THESE TO YOUR ACTUAL PATHS
    dataset_paths = {
        'depression': '/content/drive/MyDrive/Research on BD NYU /development/multi class depression dataset',
        'bipolar': '/content/drive/MyDrive/Research on BD NYU /development/Kaggle_BD ',
        'wesad': '/content/drive/MyDrive/Research on BD NYU /development/WESAD',
        'mosei': '/content/drive/MyDrive/Research on BD NYU/development/CMU-MOSEI'
    }

    print("🚀 STARTING FIXED PREPROCESSING PIPELINE")
    print("="*60)

    # Initialize components
    feature_extractor = FixedFeatureExtractor()
    loader = FixedDatasetLoader(feature_extractor)

    all_samples = []
    processing_summary = {}

    # Process each dataset
    for dataset_name, path in dataset_paths.items():
        print(f"\n{'='*50}")
        print(f"PROCESSING {dataset_name.upper()}")
        print(f"{'='*50}")

        if not os.path.exists(path):
            print(f"⚠️ Skipping {dataset_name}: path not found - {path}")
            processing_summary[dataset_name] = {'status': 'path_not_found', 'samples': 0}
            continue

        try:
            if dataset_name == 'depression':
                samples = loader.load_depression_dataset(path)
            elif dataset_name == 'bipolar':
                samples = loader.load_bipolar_dataset(path)
            elif dataset_name == 'wesad':
                samples = loader.load_wesad_dataset(path)
            elif dataset_name == 'mosei':
                samples = loader.load_cmu_mosei_dataset(path)
            else:
                samples = []

            all_samples.extend(samples)
            processing_summary[dataset_name] = {
                'status': 'success',
                'samples': len(samples)
            }
            print(f"✅ {dataset_name}: {len(samples)} samples")

        except Exception as e:
            print(f"❌ Error processing {dataset_name}: {e}")
            processing_summary[dataset_name] = {
                'status': 'error',
                'samples': 0,
                'error': str(e)
            }
            continue

    # Final summary
    print(f"\n{'='*60}")
    print("PROCESSING SUMMARY")
    print(f"{'='*60}")

    total_samples = len(all_samples)
    source_distribution = {}
    label_distribution = {}

    for sample in all_samples:
        source = sample['source']
        label = sample['label']

        source_distribution[source] = source_distribution.get(source, 0) + 1
        label_distribution[label] = label_distribution.get(label, 0) + 1

    print(f"📊 Total samples: {total_samples}")
    print(f"📈 Source distribution: {source_distribution}")
    print(f"🏷️ Label distribution: {label_distribution}")

    # Calculate class imbalance
    if label_distribution:
        max_class = max(label_distribution.values())
        min_class = min(label_distribution.values())
        imbalance_ratio = max_class / min_class if min_class > 0 else float('inf')
        print(f"⚖️ Class imbalance ratio: {imbalance_ratio:.2f}")

        if imbalance_ratio > 3.0:
            print("⚠️ SIGNIFICANT CLASS IMBALANCE DETECTED - Consider balancing techniques")

    if total_samples == 0:
        print("❌ No samples processed successfully!")
        return None, None

    # Save processed data
    print(f"\n{'='*50}")
    print("SAVING DATA")
    print(f"{'='*50}")

    output_path = '/content/drive/MyDrive/processed_mental_health_data_fixed.pkl'
    with open(output_path, 'wb') as f:
        pickle.dump(all_samples, f)

    # Save metadata
    metadata = {
        'total_samples': total_samples,
        'source_distribution': source_distribution,
        'label_distribution': label_distribution,
        'processing_summary': processing_summary,
        'feature_dimensions': {
            'text': 768,
            'audio': 88,
            'video': 2048
        },
        'num_classes': 3
    }

    metadata_path = '/content/drive/MyDrive/processing_metadata.pkl'
    with open(metadata_path, 'wb') as f:
        pickle.dump(metadata, f)

    print(f"✅ Data saved to: {output_path}")
    print(f"✅ Metadata saved to: {metadata_path}")
    print(f"✅ Processing complete!")

    return all_samples, metadata


In [ ]:
try:
    samples, metadata = run_fixed_preprocessing_pipeline()

    if samples:
        print(f"\n🎉 SUCCESS!")
        print(f"Processed {len(samples)} total samples")
        print("All four datasets should now be included if the files exist and are readable.")

except Exception as e:
    print(f"\n❌ Pipeline failed: {e}")
    import traceback
    traceback.print_exc()

🚀 STARTING FIXED PREPROCESSING PIPELINE
Loading feature extraction models...
✓ BERT loaded successfully
✓ OpenSMILE loaded successfully

PROCESSING DEPRESSION
Loading depression dataset from: /content/drive/MyDrive/Research on BD NYU /development/multi class depression dataset
Processing: /content/drive/MyDrive/Research on BD NYU /development/multi class depression dataset/dataset.csv
✓ Loaded with utf-8 encoding
Shape: (14996, 2), Columns: ['Tweets', 'Labels']
✓ Processed 14983 valid samples from /content/drive/MyDrive/Research on BD NYU /development/multi class depression dataset/dataset.csv
Processing: /content/drive/MyDrive/Research on BD NYU /development/multi class depression dataset/dataset.csv
✓ Loaded with utf-8 encoding
Shape: (14996, 2), Columns: ['Tweets', 'Labels']
✓ Processed 14983 valid samples from /content/drive/MyDrive/Research on BD NYU /development/multi class depression dataset/dataset.csv
Processing: /content/drive/MyDrive/Research on BD NYU /development/multi cla

[2025-07-20 14:54:30.527] | Success | <glove_vectors> computational sequence data in correct format.
[2025-07-20 14:54:30.527] | Status  | Checking the format of the metadata in <glove_vectors> computational sequence ...
[2025-07-20 14:54:30.527] | Warning | <glove_vectors> computational sequence does not have all the required metadata ... continuing 
[2025-07-20 14:54:30.861] | Success | Computational sequence read from file /content/drive/MyDrive/Research on BD NYU/development/CMU-MOSEI/CMU_MOSEI_COVAREP.csd ...
[2025-07-20 14:55:55.011] | Status  | Checking the integrity of the <COVAREP> computational sequence ...
[2025-07-20 14:55:55.011] | Status  | Checking the format of the data in <COVAREP> computational sequence ...


[2025-07-20 14:56:12.438] | Success | <COVAREP> computational sequence data in correct format.
[2025-07-20 14:56:12.438] | Status  | Checking the format of the metadata in <COVAREP> computational sequence ...
[2025-07-20 14:56:12.438] | Warning | <COVAREP> computational sequence does not have all the required metadata ... continuing 
[2025-07-20 14:56:12.447] | Success | Computational sequence read from file /content/drive/MyDrive/Research on BD NYU/development/CMU-MOSEI/CMU_MOSEI_VisualOpenFace2.csd ...
[2025-07-20 14:56:12.691] | Status  | Checking the integrity of the <OpenFace_2> computational sequence ...
[2025-07-20 14:56:12.691] | Status  | Checking the format of the data in <OpenFace_2> computational sequence ...


[2025-07-20 14:56:14.956] | Success | <OpenFace_2> computational sequence data in correct format.
[2025-07-20 14:56:14.956] | Status  | Checking the format of the metadata in <OpenFace_2> computational sequence ...
[2025-07-20 14:56:14.956] | Warning | <OpenFace_2> computational sequence does not have all the required metadata ... continuing 
[2025-07-20 14:56:14.969] | Success | Computational sequence read from file /content/drive/MyDrive/Research on BD NYU/development/CMU-MOSEI/CMU_MOSEI_VisualFacet42.csd ...
[2025-07-20 14:56:15.335] | Status  | Checking the integrity of the <FACET 4.2> computational sequence ...
[2025-07-20 14:56:15.335] | Status  | Checking the format of the data in <FACET 4.2> computational sequence ...


[2025-07-20 14:56:17.559] | Success | <FACET 4.2> computational sequence data in correct format.
[2025-07-20 14:56:17.559] | Status  | Checking the format of the metadata in <FACET 4.2> computational sequence ...
[2025-07-20 14:56:17.559] | Warning | <FACET 4.2> computational sequence does not have all the required metadata ... continuing 
[2025-07-20 14:56:17.564] | Success | Computational sequence read from file /content/drive/MyDrive/Research on BD NYU/development/CMU-MOSEI/CMU_MOSEI_Labels.csd ...
[2025-07-20 14:56:17.796] | Status  | Checking the integrity of the <All Labels> computational sequence ...
[2025-07-20 14:56:17.796] | Status  | Checking the format of the data in <All Labels> computational sequence ...


[2025-07-20 14:56:19.263] | Success | <All Labels> computational sequence data in correct format.
[2025-07-20 14:56:19.263] | Status  | Checking the format of the metadata in <All Labels> computational sequence ...
[2025-07-20 14:56:19.263] | Warning | <All Labels> computational sequence does not have all the required metadata ... continuing 
[2025-07-20 14:56:19.263] | Success | Dataset initialized successfully ... 
✓ CMU-MOSEI dataset initialized successfully using mmdatasdk.
Found 3293 video IDs in MOSEI.
✓ Processed 3293 valid samples from CMU-MOSEI.
✓ Total CMU-MOSEI samples: 3293
✅ mosei: 3293 samples

PROCESSING SUMMARY
📊 Total samples: 158949
📈 Source distribution: {'depression': 59932, 'bipolar': 3372, 'wesad': 92352, 'mosei': 3293}
🏷️ Label distribution: {1: 74735, 0: 44961, 2: 39253}
⚖️ Class imbalance ratio: 1.90

SAVING DATA
✅ Data saved to: /content/drive/MyDrive/processed_mental_health_data_fixed.pkl
✅ Metadata saved to: /content/drive/MyDrive/processing_metadata.pkl
✅ P

In [ ]:
!git clone https://github.com/CMU-MultiComp-Lab/CMU-MultimodalSDK.git
%cd CMU-MultimodalSDK
!pip install -r requirements.txt
!pip install .

fatal: destination path 'CMU-MultimodalSDK' already exists and is not an empty directory.
/content/CMU-MultimodalSDK
Processing /content/CMU-MultimodalSDK
  Preparing metadata (setup.py) ... done
  Created wheel for mmsdk: filename=mmsdk-1.1.0-py3-none-any.whl size=79210 sha256=93aa33ea3361d09c13aec3e6276608745e758b9ae8e1a3c76ada1189beafadc8
  Stored in directory: /root/.cache/pip/wheels/95/67/7d/7051855c4c5b9ee1d6fd0623588d84f217296da3b33953a1ab
Successfully built mmsdk
  Attempting uninstall: mmsdk
    Found existing installation: mmsdk 1.1.0
    Uninstalling mmsdk-1.1.0:
      Successfully uninstalled mmsdk-1.1.0


In [ ]:
import sys
sys.path.append("/content/CMU-MultimodalSDK")
from mmsdk import mmdatasdk
print("✅ mmsdk installed successfully!")


✅ mmsdk installed successfully!


In [ ]:
# Install CMU-MultimodalSDK if not already installed
try:
    import mmsdk
except ImportError:
    print("CMU-MultimodalSDK not found. Attempting to install CMU-MultimodalSDK...")
    !pip install CMU-MultimodalSDK
    # Attempt to import again after installation to confirm
    try:
        import mmsdk
        print("✓ CMU-MultimodalSDK installed and loaded successfully.")
    except ImportError:
        print("❌ Failed to install CMU-MultimodalSDK. Please try manual installation or check your Python version compatibility.")

In [ ]:
import os
import glob
import pandas as pd
import torch
import pickle
import numpy as np
import h5py # Assuming h5py is installed for .h5 files

# --- Minimal FeatureExtractor for testing ---
class MinimalFeatureExtractor:
    def extract_text_features(self, text):
        # Returns a dummy tensor for text features (BERT dimension)
        return torch.zeros(768)

    def extract_audio_features(self, audio_data):
        # Returns a dummy tensor for audio features (OpenSMILE dimension)
        return torch.zeros(88)

# --- FixedDatasetLoader (CMU-MOSEI only) ---
class FixedDatasetLoader:
    def __init__(self, feature_extractor):
        self.feature_extractor = feature_extractor

    def load_cmu_mosei_dataset(self, data_path):
        """Loads and processes CMU-MOSEI dataset using CMU-MultimodalSDK."""
        print(f"Loading CMU-MOSEI dataset from: {data_path}")
        samples = []

        try:
            # Import mmdatasdk here to ensure it's available after installation
            from mmsdk import mmdatasdk
            print("Attempting to load CMU-MOSEI dataset with CMU-MultimodalSDK...")

            # Load the entire dataset
            # This expects the 'data_path' to be the root directory containing CSD files
            cmu_mosei = mmdatasdk.mmdataset(data_path)
            print("✓ CMU-MOSEI dataset initialized successfully using mmdatasdk.")

            # Correctly reference the label computational sequence name
            if 'All Labels' not in cmu_mosei.computational_sequences:
                print("❌ 'All Labels' not found in MOSEI dataset. Cannot process.")
                return samples

            video_ids = list(cmu_mosei.computational_sequences['All Labels'].keys())
            print(f"Found {len(video_ids)} video IDs in MOSEI.")

            valid_count = 0
            for vid_id in video_ids:
                try:
                    # Get labels from 'All Labels' computational sequence
                    label_seq = cmu_mosei.computational_sequences['All Labels'][vid_id]['features']
                    if len(label_seq) == 0:
                        continue

                    # CMU-MOSEI labels are typically in the range [-3, 3] for sentiment
                    # We need to map them to 0 (negative), 1 (neutral), 2 (positive)
                    sentiment_score = float(label_seq[0][0]) # Assuming first score in the sequence

                    if sentiment_score < -0.5: # Adjust thresholds as needed
                        mapped_label = 0  # Negative
                    elif sentiment_score > 0.5:
                        mapped_label = 2  # Positive
                    else:
                        mapped_label = 1  # Neutral

                    # Placeholder for text, audio, and video features
                    # To fully utilize MOSEI, you would extract features from
                    # 'CMU_MOSEI_TimestampedWordVectors', 'CMU_MOSEI_COVAREP',
                    # 'CMU_MOSEI_VisualOpenFace2', 'CMU_MOSEI_VisualFacet42'
                    # and potentially align them and process with your feature_extractor.
                    # For now, we use zero tensors as placeholders.
                    text_feat = self.feature_extractor.extract_text_features("") # Pass empty string, as we're not doing actual extraction here
                    audio_feat = self.feature_extractor.extract_audio_features(None) # Pass None
                    video_feat = torch.zeros(2048) # Direct placeholder as no video extractor in MinimalFeatureExtractor

                    samples.append({
                        'user_id': str(vid_id),
                        'text': text_feat,
                        'audio': audio_feat,
                        'video': video_feat,
                        'label': mapped_label,
                        'source': 'mosei'
                    })
                    valid_count += 1

                except Exception as e:
                    print(f"Error processing video ID {vid_id}: {e}")
                    continue

            print(f"✓ Processed {valid_count} valid samples from CMU-MOSEI.")

        except ImportError:
            print("❌ CMU-MultimodalSDK not found. Please ensure it is installed correctly to process CSD files.")
            print("You might need to run the following commands in a separate cell at the beginning of your notebook:")
            print("!git clone https://github.com/CMU-MultiComp-Lab/CMU-MultimodalSDK.git")
            print("%cd CMU-MultimodalSDK")
            print("!pip install -r requirements.txt")
            print("!pip install .")
            print("import sys")
            print("sys.path.append(\"/content/CMU-MultimodalSDK\")")

        except Exception as e:
            print(f"❌ Error loading CMU-MOSEI dataset with mmdatasdk: {e}")

        print(f"✓ Total CMU-MOSEI samples: {len(samples)}")
        return samples

    # Helper methods (kept for completeness if needed, but not used by MOSEI-only loader)
    def _find_column(self, columns, keywords):
        columns_lower = [col.lower() for col in columns]
        for keyword in keywords:
            for col, col_lower in zip(columns, columns_lower):
                if keyword.lower() in col_lower:
                    return col
        return None

# --- How to test for CMU-MOSEI ---
if __name__ == '__main__':
    # 1. IMPORTANT: Ensure CMU-MultimodalSDK is installed FIRST.
    #    Run these lines in a separate cell AT THE VERY BEGINNING of your Colab notebook:
    #    !git clone https://github.com/CMU-MultiComp-Lab/CMU-MultimodalSDK.git
    #    %cd CMU-MultimodalSDK
    #    !pip install -r requirements.txt
    #    !pip install .
    #    import sys
    #    sys.path.append("/content/CMU-MultimodalSDK")
    #    # Then, you can run the rest of the code in another cell.

    # 2. Define the path to your CMU-MOSEI dataset
    #    Make sure this path is correct and accessible (e.g., Google Drive mounted)
    cmu_mosei_data_path = "/content/drive/MyDrive/Research on BD NYU/development/CMU-MOSEI"

    # 3. Initialize the MinimalFeatureExtractor
    feature_extractor = MinimalFeatureExtractor()

    # 4. Initialize the FixedDatasetLoader with the feature extractor
    loader = FixedDatasetLoader(feature_extractor)

    # 5. Load the CMU-MOSEI dataset
    mosei_samples = loader.load_cmu_mosei_dataset(cmu_mosei_data_path)

    # 6. Print some summary information
    print("\n--- CMU-MOSEI Processing Summary ---")
    print(f"Total CMU-MOSEI samples loaded: {len(mosei_samples)}")

    if len(mosei_samples) > 0:
        print("First 3 samples (user_id and label):")
        for i, sample in enumerate(mosei_samples[:3]):
            print(f"  Sample {i+1}: User ID: {sample['user_id']}, Label: {sample['label']}")

        # Verify feature dimensions (should be dummy zeros)
        print(f"Example text feature shape: {mosei_samples[0]['text'].shape}")
        print(f"Example audio feature shape: {mosei_samples[0]['audio'].shape}")
        print(f"Example video feature shape: {mosei_samples[0]['video'].shape}")

Loading CMU-MOSEI dataset from: /content/drive/MyDrive/Research on BD NYU/development/CMU-MOSEI
Attempting to load CMU-MOSEI dataset with CMU-MultimodalSDK...
[2025-07-20 13:58:45.387] | Success | Computational sequence read from file /content/drive/MyDrive/Research on BD NYU/development/CMU-MOSEI/CMU_MOSEI_TimestampedWordVectors.csd ...
[2025-07-20 13:58:45.686] | Status  | Checking the integrity of the <glove_vectors> computational sequence ...
[2025-07-20 13:58:45.686] | Status  | Checking the format of the data in <glove_vectors> computational sequence ...


[2025-07-20 13:58:48.003] | Success | <glove_vectors> computational sequence data in correct format.
[2025-07-20 13:58:48.003] | Status  | Checking the format of the metadata in <glove_vectors> computational sequence ...
[2025-07-20 13:58:48.003] | Warning | <glove_vectors> computational sequence does not have all the required metadata ... continuing 
[2025-07-20 13:58:48.012] | Success | Computational sequence read from file /content/drive/MyDrive/Research on BD NYU/development/CMU-MOSEI/CMU_MOSEI_COVAREP.csd ...
[2025-07-20 13:58:48.274] | Status  | Checking the integrity of the <COVAREP> computational sequence ...
[2025-07-20 13:58:48.274] | Status  | Checking the format of the data in <COVAREP> computational sequence ...


[2025-07-20 13:58:50.426] | Success | <COVAREP> computational sequence data in correct format.
[2025-07-20 13:58:50.426] | Status  | Checking the format of the metadata in <COVAREP> computational sequence ...
[2025-07-20 13:58:50.426] | Warning | <COVAREP> computational sequence does not have all the required metadata ... continuing 
[2025-07-20 13:58:50.435] | Success | Computational sequence read from file /content/drive/MyDrive/Research on BD NYU/development/CMU-MOSEI/CMU_MOSEI_VisualOpenFace2.csd ...
[2025-07-20 13:58:50.797] | Status  | Checking the integrity of the <OpenFace_2> computational sequence ...
[2025-07-20 13:58:50.797] | Status  | Checking the format of the data in <OpenFace_2> computational sequence ...


[2025-07-20 13:58:53.140] | Success | <OpenFace_2> computational sequence data in correct format.
[2025-07-20 13:58:53.140] | Status  | Checking the format of the metadata in <OpenFace_2> computational sequence ...
[2025-07-20 13:58:53.140] | Warning | <OpenFace_2> computational sequence does not have all the required metadata ... continuing 
[2025-07-20 13:58:53.153] | Success | Computational sequence read from file /content/drive/MyDrive/Research on BD NYU/development/CMU-MOSEI/CMU_MOSEI_VisualFacet42.csd ...
[2025-07-20 13:58:53.491] | Status  | Checking the integrity of the <FACET 4.2> computational sequence ...
[2025-07-20 13:58:53.491] | Status  | Checking the format of the data in <FACET 4.2> computational sequence ...


[2025-07-20 13:58:55.927] | Success | <FACET 4.2> computational sequence data in correct format.
[2025-07-20 13:58:55.927] | Status  | Checking the format of the metadata in <FACET 4.2> computational sequence ...
[2025-07-20 13:58:55.927] | Warning | <FACET 4.2> computational sequence does not have all the required metadata ... continuing 
[2025-07-20 13:58:55.933] | Success | Computational sequence read from file /content/drive/MyDrive/Research on BD NYU/development/CMU-MOSEI/CMU_MOSEI_Labels.csd ...
[2025-07-20 13:58:56.201] | Status  | Checking the integrity of the <All Labels> computational sequence ...
[2025-07-20 13:58:56.201] | Status  | Checking the format of the data in <All Labels> computational sequence ...


[2025-07-20 13:58:57.674] | Success | <All Labels> computational sequence data in correct format.
[2025-07-20 13:58:57.674] | Status  | Checking the format of the metadata in <All Labels> computational sequence ...
[2025-07-20 13:58:57.674] | Warning | <All Labels> computational sequence does not have all the required metadata ... continuing 
[2025-07-20 13:58:57.674] | Success | Dataset initialized successfully ... 
✓ CMU-MOSEI dataset initialized successfully using mmdatasdk.
Found 3293 video IDs in MOSEI.
✓ Processed 3293 valid samples from CMU-MOSEI.
✓ Total CMU-MOSEI samples: 3293

--- CMU-MOSEI Processing Summary ---
Total CMU-MOSEI samples loaded: 3293
First 3 samples (user_id and label):
  Sample 1: User ID: --qXJuDtHPw, Label: 2
  Sample 2: User ID: -3g5yACwYnA, Label: 2
  Sample 3: User ID: -3nNcZdcdvU, Label: 1
Example text feature shape: torch.Size([768])
Example audio feature shape: torch.Size([88])
Example video feature shape: torch.Size([2048])
